In [ ]:
import os

print(os.listdir("/kaggle/input"))


In [ ]:
print(os.listdir("/kaggle/input/multi-fruit-leaf-disease-dataset"))


In [ ]:
print(os.listdir("/kaggle/input/multi-fruit-leaf-disease-dataset/Dataset"))


In [ ]:
BASE_PATH = "/kaggle/input/multi-fruit-leaf-disease-dataset/Dataset/Train"

In [ ]:
print(len(os.listdir(BASE_PATH)))
print(os.listdir(BASE_PATH)[:5])


# Let's create test and validation data

In [ ]:
import os
import shutil
import random

random.seed(42)

SRC_DIR = "/kaggle/input/multi-fruit-leaf-disease-dataset/Dataset/Train"
DEST_DIR = "/kaggle/working/Dataset"

TRAIN_DIR = os.path.join(DEST_DIR, "train")
VAL_DIR   = os.path.join(DEST_DIR, "val")
TEST_DIR  = os.path.join(DEST_DIR, "test")

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(d, exist_ok=True)


In [ ]:
train_ratio = 0.7
val_ratio = 0.15

for class_name in os.listdir(SRC_DIR):
    class_path = os.path.join(SRC_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(train_ratio * total)
    val_end = train_end + int(val_ratio * total)

    split_map = {
        TRAIN_DIR: images[:train_end],
        VAL_DIR: images[train_end:val_end],
        TEST_DIR: images[val_end:]
    }

    for split_dir, img_list in split_map.items():
        split_class_dir = os.path.join(split_dir, class_name)
        os.makedirs(split_class_dir, exist_ok=True)

        for img in img_list:
            shutil.copy(
                os.path.join(class_path, img),
                os.path.join(split_class_dir, img)
            )


In [ ]:
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    split_path = f"/kaggle/working/Dataset/{split}"
    for cls in os.listdir(split_path):
        count = len(os.listdir(os.path.join(split_path, cls)))
        print(f"{cls}: {count}")


# Problem 1: EXTREME CLASS IMBALANCE (major issue)
Model will become “healthy-biased”
Rare diseases will have terrible recall
# Problem 2: Very tiny validation & test sets for some classes
banana__leaf__disease__scab_moth

Val = 3
Test = 3

One wrong prediction = 33% accuracy drop
Metrics will be unstable

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))


In [ ]:
TRAIN_DIR = "/kaggle/working/Dataset/train"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

class_names = train_ds.class_names
print("Total classes:", len(class_names))


In [ ]:
plt.figure(figsize=(20, 20))

shown = set()
i = 1

for images, labels in train_ds:
    for img, label in zip(images, labels):
        class_name = class_names[label.numpy()]
        
        if class_name not in shown:
            plt.subplot(5, 4, i)
            plt.imshow(img.numpy().astype("uint8"))
            plt.title(class_name, fontsize=9)
            plt.axis("off")
            
            shown.add(class_name)
            i += 1
            
        if len(shown) == len(class_names):
            break
    if len(shown) == len(class_names):
        break

plt.tight_layout()
plt.show()


In [ ]:
IMG_SIZE = (240, 240)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/Dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/Dataset/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/Dataset/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
NUM_CLASSES = len(train_ds.class_names)
print("Classes:", NUM_CLASSES)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
])


In [ ]:
base_model = tf.keras.applications.EfficientNetB1(
    include_top=False,
    input_shape=(240, 240, 3),
    weights="imagenet"
)


In [ ]:

base_model.trainable = False  # Phase 1


In [ ]:
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = []
for _, y in train_ds:
    labels.extend(y.numpy())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3)
]

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
print("Final Train Accuracy:", history_1.history["accuracy"][-1])
print("Final Val Accuracy:", history_1.history["val_accuracy"][-1])


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks
)


# Accuracy plot (Train vs Val)

In [ ]:
acc = history_1.history["accuracy"] + history_2.history["accuracy"]
val_acc = history_1.history["val_accuracy"] + history_2.history["val_accuracy"]

plt.figure(figsize=(8, 5))
plt.plot(acc, marker='o', label="Train Accuracy")
plt.plot(val_acc, marker='o', label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy (Full Training)")
plt.legend()
plt.grid(True)
plt.show()


# Loss plot (Train vs Val)

In [ ]:
loss = history_1.history["loss"]
val_loss = history_1.history["val_loss"]

plt.figure(figsize=(8, 5))
plt.plot(loss, marker='o', label="Train Loss")
plt.plot(val_loss, marker='o', label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


# Eval on Test dataset

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")


In [ ]:
import numpy as np

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class_names = test_ds.class_names
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    xticklabels=class_names,
    yticklabels=class_names,
    annot=False,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (Test Set)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))


# Save the model

In [ ]:
MODEL_PATH = "/kaggle/working/leaf_disease_model.keras"
model.save(MODEL_PATH)
print("Model saved at:", MODEL_PATH)
# model kaggle ma store thase

In [ ]:
import json

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(train_ds.class_names, f)
# class name file
# bnne locally download krvi